# A variational autoencoder, built

Encoder to a distribution, a sampling layer, a decoder — and both halves of the loss, with an ablation showing what each one does.

**Runs on:** CPU — about 8 minutes (GPU: 2 minutes) &nbsp;·&nbsp; **Slides:** [Chapter 17 — Image Generation](../../../course-web-slides/ch17/index.html) &nbsp;·&nbsp; **Section:** 01 — Variational autoencoders

---

## First, a classical autoencoder, so the difference is visible

In [ ]:
import keras
from keras import layers
from keras.datasets import mnist
import numpy as np
import matplotlib.pyplot as plt

(x_train, _), (x_test, _) = mnist.load_data()
x_train = np.expand_dims(x_train, -1).astype("float32") / 255
x_test = np.expand_dims(x_test, -1).astype("float32") / 255

keras.utils.set_random_seed(0)
inp = keras.Input(shape=(28, 28, 1))
z = layers.Conv2D(32, 3, activation="relu", strides=2, padding="same")(inp)
z = layers.Conv2D(64, 3, activation="relu", strides=2, padding="same")(z)
z = layers.Flatten()(z)
code = layers.Dense(2, name="code")(z)           # a POINT, not a distribution
z = layers.Dense(7 * 7 * 64, activation="relu")(code)
z = layers.Reshape((7, 7, 64))(z)
z = layers.Conv2DTranspose(64, 3, activation="relu", strides=2,
                           padding="same")(z)
z = layers.Conv2DTranspose(32, 3, activation="relu", strides=2,
                           padding="same")(z)
out = layers.Conv2D(1, 3, activation="sigmoid", padding="same")(z)
plain_ae = keras.Model(inp, out)
plain_ae.compile(optimizer="adam", loss="binary_crossentropy")
plain_ae.fit(x_train, x_train, epochs=10, batch_size=128, verbose=0)
print("plain autoencoder trained")

In [ ]:
plain_encoder = keras.Model(inp, code)
codes = plain_encoder.predict(x_test[:4000], verbose=0)

plt.figure(figsize=(6, 6))
plt.scatter(codes[:, 0], codes[:, 1], c=mnist.load_data()[1][1][:4000],
            cmap="tab10", s=5, alpha=.6)
plt.colorbar(); plt.title("Plain autoencoder: the 2-d code space")
plt.show()

# Decode a grid of points and see what is between the clusters.
dec_in = keras.Input(shape=(2,))
# Start just after the code layer -- by NAME, because layer indices shift between
# Keras versions (Keras 3 counts the InputLayer, Keras 2 did not).
start = plain_ae.layers.index(plain_ae.get_layer("code")) + 1
h = dec_in
for layer in plain_ae.layers[start:]:
    h = layer(h)
plain_decoder = keras.Model(dec_in, h)

grid = np.array([[x, y] for y in np.linspace(codes[:,1].max(), codes[:,1].min(), 8)
                        for x in np.linspace(codes[:,0].min(), codes[:,0].max(), 8)])
imgs = plain_decoder.predict(grid, verbose=0)
fig, axes = plt.subplots(8, 8, figsize=(7, 7))
for ax, im in zip(axes.ravel(), imgs):
    ax.imshow(im[:, :, 0], cmap="gray_r"); ax.axis("off")
plt.suptitle("Plain autoencoder: islands, and mush between them", y=1.0)
plt.tight_layout(); plt.show()

**Islands of digits with regions between them that decode to nothing.** The space is not continuous, so sampling from it does not work — which is exactly why classical autoencoders fell out of fashion for generation.

## The VAE encoder: two outputs, not one

In [ ]:
latent_dim = 2

image_inputs = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation="relu", strides=2, padding="same")(
    image_inputs)
x = layers.Conv2D(64, 3, activation="relu", strides=2, padding="same")(x)
x = layers.Flatten()(x)
x = layers.Dense(16, activation="relu")(x)
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
encoder = keras.Model(image_inputs, [z_mean, z_log_var], name="encoder")
encoder.summary()

**Strides, not pooling** — the same reason as chapter 11. The encoding has to support reconstructing a valid image, so *where* things are must survive.

## The sampling layer

In [ ]:
from keras import ops

class Sampler(keras.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.seed_generator = keras.random.SeedGenerator()
        self.built = True

    def call(self, z_mean, z_log_var):
        batch_size = ops.shape(z_mean)[0]
        z_size = ops.shape(z_mean)[1]
        epsilon = keras.random.normal((batch_size, z_size),
                                      seed=self.seed_generator)
        return z_mean + ops.exp(0.5 * z_log_var) * epsilon

`exp(0.5 * z_log_var)` turns a **log variance** into a standard deviation. Predicting the log keeps the encoder's output unconstrained — it may be negative — and the exponential makes it positive.

## The decoder

In [ ]:
latent_inputs = keras.Input(shape=(latent_dim,))
x = layers.Dense(7 * 7 * 64, activation="relu")(latent_inputs)
x = layers.Reshape((7, 7, 64))(x)
x = layers.Conv2DTranspose(64, 3, activation="relu", strides=2,
                           padding="same")(x)
x = layers.Conv2DTranspose(32, 3, activation="relu", strides=2,
                           padding="same")(x)
decoder_outputs = layers.Conv2D(1, 3, activation="sigmoid", padding="same")(x)
decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")
decoder.summary()

A mirror of the encoder. `Dense(7*7*64)` produces exactly the coefficients the encoder's `Flatten` consumed — **reading it as a mirror is the quickest way to check it.**

## compute_loss, not train_step

In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.sampler = Sampler()
        self.reconstruction_loss_tracker = keras.metrics.Mean(
            name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    def call(self, inputs):
        return self.encoder(inputs)

    def compute_loss(self, x, y, y_pred, sample_weight=None, training=True):
        z_mean, z_log_var = y_pred
        reconstruction = self.decoder(self.sampler(z_mean, z_log_var))
        reconstruction_loss = ops.mean(ops.sum(
            keras.losses.binary_crossentropy(x, reconstruction), axis=(1, 2)))
        kl_loss = -0.5 * (1 + z_log_var - ops.square(z_mean)
                          - ops.exp(z_log_var))
        total_loss = reconstruction_loss + ops.mean(kl_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return total_loss

This is the first model in the course **not doing supervised learning** — the input is the target. Departing from supervised learning normally means a custom `train_step()`, which is **backend specific**. Overriding `compute_loss()` instead keeps the default `train_step` and runs unchanged on all three backends.

## Training, with no loss and no targets

In [ ]:
mnist_digits = np.concatenate([x_train, x_test], axis=0)

vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam())     # note: no loss=
vae.fit(mnist_digits, epochs=30, batch_size=128, verbose=2)

> **Note** — No `loss=` at compile time, and **no targets in `fit()`**. Both follow from `compute_loss()` supplying its own objective.

## Watching the two loss terms

In [ ]:
h = vae.history.history
plt.figure(figsize=(7, 4.2))
plt.plot(h["reconstruction_loss"], lw=1.6, label="reconstruction")
plt.plot(h["kl_loss"], lw=1.6, label="KL divergence")
plt.xlabel("epoch"); plt.legend(); plt.yscale("log")
plt.title("The two terms pull in different directions")
plt.show()

Reconstruction wants the encoder to spread points out so it can tell them apart. KL wants them collapsed onto a unit normal. **The equilibrium between the two is the structured space.**

## The ablation: what each term does

In [ ]:
class NoKL(VAE):
    def compute_loss(self, x, y, y_pred, sample_weight=None, training=True):
        z_mean, z_log_var = y_pred
        rec = self.decoder(self.sampler(z_mean, z_log_var))
        return ops.mean(ops.sum(
            keras.losses.binary_crossentropy(x, rec), axis=(1, 2)))

keras.utils.set_random_seed(0)
enc2 = keras.models.clone_model(encoder)
dec2 = keras.models.clone_model(decoder)
nokl = NoKL(enc2, dec2)
nokl.compile(optimizer=keras.optimizers.Adam())
nokl.fit(mnist_digits[:20000], epochs=10, batch_size=128, verbose=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.4))
for ax, model, title in [(axes[0], vae, "with KL term"),
                         (axes[1], nokl, "reconstruction only")]:
    m, _ = model.encoder.predict(x_test[:3000], verbose=0)
    ax.scatter(m[:, 0], m[:, 1], c=mnist.load_data()[1][1][:3000],
               cmap="tab10", s=5, alpha=.6)
    ax.set_title(title)
plt.suptitle("The KL term is what makes the space usable", y=1.0)
plt.tight_layout(); plt.show()

Without KL the encoder spreads points as far apart as it likes — reconstruction improves, and the space becomes **unsamplable**, which is the plain autoencoder again.

**The regularizer is not a refinement here. It is the entire reason the model is generative.**

---

## What to take away

- A classical autoencoder's latent space has islands; the gaps decode to nothing.
- A VAE encodes to a **distribution**, samples from it, and adds a KL term.
- Override `compute_loss()` rather than `train_step()` to stay backend-agnostic.
- Ablate the KL term and the space becomes unsamplable — it is the reason the model is generative.